# 🎮 Phase 7: AI Agent Layer - Tooling, Intent Routing & Dialogue Prototyping

> **Mục tiêu của Notebook (Task 7.1):**
> 1. **Thiết kế & Hiện thực hóa bộ 3 Core Agent Tools:**
>    - **`RecommendTool`:** Gợi ý game thông minh dựa trên Hybrid Recommender (CF + Semantic CB + Sentiment) & MMR Diversity Re-ranking cho cả User đã có lịch sử và Cold-Start User (tìm kiếm theo từ khóa/mô tả tự nhiên).
>    - **`ExplainTool`:** Giải thích minh bạch, đa chiều lý do hệ thống đề xuất một tựa game (Anchor game, Similarity %, Genre overlap, Collaborative consensus, Sentiment & Real player quote).
>    - **`AnalyticsTool`:** Phân tích toàn diện hồ sơ game thủ (Gamer Persona, Rating distribution, Favorite genres, Average rating, Activity timeline, Sentiment bias).
> 2. **Xây dựng Agent Tool Router & ReAct Orchestrator:**
>    - Nhận diện ý định người dùng (Intent Classification & Parameter Extraction) từ câu hỏi tự nhiên.
>    - Tự động điều phối (Routing) gọi đúng Tool tương ứng hoặc kết hợp chuỗi Tool (Tool Chaining).
>    - Sinh câu trả lời mượt mà, chuyên nghiệp, giàu thông tin và mang tính cá nhân hóa cao.
> 3. **Thử nghiệm các Kịch bản Đối thoại Thực tế (End-to-End Dialogue Simulation):**
>    - *Kịch bản 1:* Phân tích gu và lịch sử chơi game của User.
>    - *Kịch bản 2:* Gợi ý game theo sở thích cá nhân hóa của User cũ.
>    - *Kịch bản 3:* Gợi ý theo mô tả ngôn ngữ tự nhiên cho Cold-Start User ("Tìm game Mario phiêu lưu hay nhất").
>    - *Kịch bản 4:* Giải thích chi tiết một đề xuất cụ thể kèm trích dẫn thực tế từ cộng đồng.

In [ ]:
import os
import sys
import json
import re
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Optional, Any, Union

# Đảm bảo import được các module trong src/
sys.path.append("..")
from src.models.content_based.recommender import ContentBasedRecommender
from src.models.collaborative.matrix_factorization import SVDRecommender
from src.models.hybrid.hybrid_engine import HybridRecommender, HybridRecommendationResult
from src.models.hybrid.explainer import RecommendationExplainer, RecommendationExplanation
from src.models.ranking import DiversityRanker

print("[*] Core ML/NLP modules imported successfully!")

## 1. Tải Dữ liệu & Khởi tạo Hệ thống Nền tảng (Recommender Engines)

In [ ]:
# 1. Khởi tạo Content-Based Model
cb_model = ContentBasedRecommender(
    embeddings_path="../data/gold/item_embeddings.npy",
    items_path="../data/silver/item_features.parquet"
)

# 2. Khởi tạo SVD Collaborative Filtering Model
svd_model = SVDRecommender.load_model("../models/collaborative/svd_recommender.joblib")

# 3. Tải Dữ liệu Sentiment & Reviews & Interactions
df_sentiment = pl.read_parquet("../data/silver/item_sentiment.parquet")
df_reviews = pl.read_parquet("../data/silver/review_sentiment.parquet")
df_interactions = pl.read_parquet("../data/silver/interactions.parquet")

# 4. Khởi tạo Hybrid Engine, Explainer & Diversity Ranker
hybrid_engine = HybridRecommender(
    cb_model=cb_model,
    svd_model=svd_model,
    df_sentiment=df_sentiment,
    df_interactions=df_interactions,
)

explainer = RecommendationExplainer(
    cb_model=cb_model,
    df_sentiment=df_sentiment,
    df_reviews=df_reviews,
)

ranker = DiversityRanker(
    embeddings=cb_model.embeddings,
    item2idx=cb_model.item2idx,
)

print(f"[+] Hybrid Recommender initialized with {len(cb_model.item_ids):,} items and {len(svd_model.user2idx):,} CF users.")

---

## 2. Thiết kế & Hiện thực hóa Agent Core Tools

Mỗi Tool của Agent được thiết kế theo chuẩn:
- Có **Schema / Input Specification** rõ ràng.
- Trả về kết quả có cấu trúc (**Structured Output**) và sẵn sàng để LLM đọc / định dạng thành phản hồi tự nhiên.

### 🛠️ 2.1. Tool 1: `RecommendTool` (Gợi ý Game Thông Minh)
Hỗ trợ cả:
1. **Personalized Hybrid Recommendation:** Gợi ý theo `user_id` với SVD CF + Centroid Embeddings + VADER Sentiment + MMR Re-ranking.
2. **Semantic / Keyword Search Recommendation:** Cho User mới hoặc tìm kiếm tự do theo thể loại/từ khóa.

In [ ]:
def find_items_by_keyword(cb: ContentBasedRecommender, keyword: str, top_k: int = 3) -> List[Dict[str, Any]]:
    """
    Tìm kiếm tựa game theo từ khóa/tên game kết hợp Exact Substring và Token Matching.
    """
    kw = keyword.lower().strip()
    matches = []
    # 1. Exact substring match
    for asin, title in cb.item_titles.items():
        if kw in title.lower():
            matches.append({
                "parent_asin": asin,
                "title": title,
                "category": cb.item_categories.get(asin, "Video Games"),
            })
            if len(matches) >= top_k:
                return matches

    # 2. Token-based fallback (tách từ đơn lẻ có nghĩa)
    tokens = [t for t in re.findall(r"\w+", kw) if len(t) >= 3 and t not in ["game", "play", "cho", "hay", "nhất"]]
    for token in tokens:
        for asin, title in cb.item_titles.items():
            if token in title.lower() and not any(m["parent_asin"] == asin for m in matches):
                matches.append({
                    "parent_asin": asin,
                    "title": title,
                    "category": cb.item_categories.get(asin, "Video Games"),
                })
                if len(matches) >= top_k:
                    return matches

    return matches

class RecommendTool:
    """
    Tool gợi ý game thông minh cho AI Agent.
    """
    name = "recommend_games"
    description = (
        "Gợi ý danh sách game phù hợp nhất dựa trên mã người dùng (user_id), "
        "hoặc tìm kiếm ngữ nghĩa theo từ khóa/thể loại/tựa game yêu thích. "
        "Tự động áp dụng MMR Re-ranking để tăng tính đa dạng danh mục."
    )

    def __init__(self, hybrid_engine: HybridRecommender, ranker: DiversityRanker, explainer: RecommendationExplainer):
        self.hybrid_engine = hybrid_engine
        self.ranker = ranker
        self.explainer = explainer

    def run(
        self,
        user_id: Optional[str] = None,
        query: Optional[str] = None,
        category: Optional[str] = None,
        top_k: int = 5,
        diversity_weight: float = 0.3,
        include_explanation: bool = True,
    ) -> Dict[str, Any]:
        """
        Thực thi gợi ý game.
        """
        # 1. Trường hợp tìm kiếm theo câu truy vấn / từ khóa (Semantic Search)
        if query and not user_id:
            matched_items = find_items_by_keyword(self.hybrid_engine.cb_model, query, top_k=3)
            if matched_items:
                anchor_asin = matched_items[0]["parent_asin"]
                anchor_title = matched_items[0]["title"]
                raw_recs = self.hybrid_engine.recommend(
                    liked_item_ids=[anchor_asin],
                    top_k=top_k * 2,
                    category_filter=category,
                )
            else:
                raw_recs = self.hybrid_engine.recommend(
                    liked_item_ids=None,
                    top_k=top_k * 2,
                    category_filter=category,
                )
                anchor_asin, anchor_title = None, None
                
            # Áp dụng MMR Re-ranking đa dạng hóa
            ranked_items = self.ranker.apply_mmr(raw_recs.items, top_k=top_k, lambda_param=1.0 - diversity_weight)
            
            results = []
            for item in ranked_items:
                exp = self.explainer.explain_recommendation(
                    item, 
                    user_liked_asins=[anchor_asin] if anchor_asin else [],
                    is_cold_start=True
                )
                item_dict = item.to_dict()
                if include_explanation:
                    item_dict["explanation"] = exp.to_dict()
                results.append(item_dict)
                
            return {
                "status": "success",
                "mode": "semantic_query",
                "query": query,
                "matched_anchor": anchor_title,
                "category_filter": category,
                "total_recommendations": len(results),
                "recommendations": results,
            }

        # 2. Trường hợp gợi ý cá nhân hóa theo User ID
        elif user_id:
            user_history = self.hybrid_engine.user_history_dict.get(user_id, [])
            raw_output = self.hybrid_engine.recommend(
                user_id=user_id,
                top_k=top_k * 2,
                category_filter=category,
            )
            
            ranked_items = self.ranker.apply_mmr(raw_output.items, top_k=top_k, lambda_param=1.0 - diversity_weight)
            
            results = []
            for item in ranked_items:
                exp = self.explainer.explain_recommendation(
                    item,
                    user_liked_asins=user_history,
                    is_cold_start=False
                )
                item_dict = item.to_dict()
                if include_explanation:
                    item_dict["explanation"] = exp.to_dict()
                results.append(item_dict)
                
            return {
                "status": "success",
                "mode": "personalized_user",
                "user_id": user_id,
                "user_history_count": len(user_history),
                "cf_available": raw_output.cf_available,
                "effective_weights": raw_output.effective_weights,
                "total_recommendations": len(results),
                "recommendations": results,
            }
            
        else:
            # Gợi ý mặc định theo game đánh giá cao
            raw_recs = self.hybrid_engine.recommend(top_k=top_k, category_filter=category)
            return {
                "status": "success",
                "mode": "popular_top_rated",
                "recommendations": raw_recs.to_dict_list(),
            }

recommend_tool = RecommendTool(hybrid_engine, ranker, explainer)
print("[*] RecommendTool initialized successfully!")

### 🛠️ 2.2. Tool 2: `ExplainTool` (Giải Thích Chi Tiết Đề Xuất)
Trích xuất các căn cứ logic, sự tương đồng nội dung, phản hồi cộng đồng và đánh giá xác thực của game thủ.

In [ ]:
class ExplainTool:
    """
    Tool giải thích lý do gợi ý game cho AI Agent.
    """
    name = "explain_recommendation"
    description = (
        "Giải thích chi tiết vì sao một tựa game được đề xuất cho người dùng. "
        "Cung cấp mức độ tương đồng với game đã chơi trong quá khứ, trùng khớp thể loại, "
        "và trích dẫn nhận xét tích cực thực tế từ cộng đồng game thủ."
    )

    def __init__(self, explainer: RecommendationExplainer, hybrid_engine: HybridRecommender):
        self.explainer = explainer
        self.hybrid_engine = hybrid_engine

    def run(
        self,
        item_asin_or_title: str,
        user_id: Optional[str] = None,
        reference_game_title: Optional[str] = None,
    ) -> Dict[str, Any]:
        """
        Thực thi giải thích lý do gợi ý.
        """
        asin = item_asin_or_title
        if asin not in self.explainer.cb_model.item2idx:
            found = find_items_by_keyword(self.explainer.cb_model, item_asin_or_title, top_k=1)
            if found:
                asin = found[0]["parent_asin"]
            else:
                return {
                    "status": "error",
                    "message": f"Không tìm thấy thông tin tựa game với mã hoặc tên: '{item_asin_or_title}'"
                }

        title = self.explainer.cb_model.item_titles.get(asin, "Unknown Title")
        category = self.explainer.cb_model.item_categories.get(asin, "Video Games")
        rating = float(self.explainer.cb_model.item_ratings.get(asin, 4.5))

        # Lấy lịch sử user nếu có
        user_history = []
        if user_id:
            user_history = self.hybrid_engine.user_history_dict.get(user_id, [])
        elif reference_game_title:
            ref_found = find_items_by_keyword(self.explainer.cb_model, reference_game_title, top_k=1)
            if ref_found:
                user_history = [ref_found[0]["parent_asin"]]

        # Tạo dummy HybridRecommendationResult để truyền vào explainer
        item_res = HybridRecommendationResult(
            parent_asin=asin,
            title=title,
            category=category,
            hybrid_score=0.9,
            cf_score=0.85,
            cb_score=0.92,
            sentiment_score=0.88,
            avg_rating=rating,
            rating_number=100,
        )

        explanation = self.explainer.explain_recommendation(
            item_res, 
            user_liked_asins=user_history,
            is_cold_start=(user_id is None)
        )
        
        return {
            "status": "success",
            "parent_asin": asin,
            "title": title,
            "category": category,
            "average_rating": rating,
            "anchor_game": explanation.anchor_title,
            "anchor_similarity_pct": round(explanation.anchor_similarity * 100, 1),
            "key_reasons": explanation.reasons,
            "social_proof_quote": explanation.highlight_quote,
        }

explain_tool = ExplainTool(explainer, hybrid_engine)
print("[*] ExplainTool initialized successfully!")

### 🛠️ 2.3. Tool 3: `AnalyticsTool` (Phân Tích Hồ Sơ & Hành Vi Game Thủ)
Tổng hợp toàn diện lịch sử tương tác, phân bố điểm đánh giá, thể loại ưa thích nhất và hành vi tiêu dùng của User.

In [ ]:
class AnalyticsTool:
    """
    Tool phân tích hồ sơ và thống kê hành vi người dùng cho AI Agent.
    """
    name = "analyze_user_profile"
    description = (
        "Phân tích chi tiết lịch sử chơi game, phân bố đánh giá (1-5 sao), "
        "các thể loại yêu thích nhất, và xây dựng Gamer Persona của người dùng."
    )

    def __init__(self, df_interactions: pl.DataFrame, cb_model: ContentBasedRecommender):
        self.df_interactions = df_interactions
        self.cb_model = cb_model

    def run(self, user_id: str) -> Dict[str, Any]:
        """
        Thực thi phân tích hồ sơ người dùng.
        """
        # Lọc lịch sử của user
        user_df = self.df_interactions.filter(pl.col("user_id") == user_id)
        
        if user_df.is_empty():
            return {
                "status": "not_found",
                "user_id": user_id,
                "message": f"Không tìm thấy lịch sử tương tác cho người dùng '{user_id}'. Đây là Cold-Start User."
            }
            
        total_reviews = len(user_df)
        avg_rating = float(user_df.select(pl.col("rating").mean()).item())
        
        # Phân bố rating
        rating_counts = user_df.group_by("rating").len().sort("rating", descending=True)
        rating_dist = {int(row["rating"]): int(row["len"]) for row in rating_counts.iter_rows(named=True)}
        
        # Lấy danh sách các game đã tương tác kèm metadata
        item_asins = user_df.select("parent_asin").to_series().to_list()
        ratings = user_df.select("rating").to_series().to_list()
        
        history_items = []
        category_counts: Dict[str, int] = {}
        
        for asin, rating in zip(item_asins, ratings):
            title = self.cb_model.item_titles.get(asin, "Unknown Title")
            cat = self.cb_model.item_categories.get(asin, "Video Games")
            
            category_counts[cat] = category_counts.get(cat, 0) + 1
            history_items.append({
                "parent_asin": asin,
                "title": title,
                "category": cat,
                "user_rating": float(rating),
            })
            
        # Sắp xếp thể loại phổ biến nhất
        sorted_categories = sorted(category_counts.items(), key=lambda x: x[1], reverse=True)
        top_categories = [cat for cat, count in sorted_categories[:3]]
        
        # Xác định Gamer Persona
        primary_cat = sorted_categories[0][0] if sorted_categories else "General Gamer"
        if "Role Playing" in primary_cat or "RPG" in primary_cat:
            persona = "RPG Enthusiast / Story Explorer"
        elif "Action" in primary_cat or "Shooter" in primary_cat:
            persona = "Action & Adrenaline Seeker"
        elif "Strategy" in primary_cat or "Simulation" in primary_cat:
            persona = "Tactical Strategist & Builder"
        elif "Retro" in primary_cat or "Classic" in primary_cat:
            persona = "Retro Gaming Collector"
        else:
            persona = f"{primary_cat} Explorer"
            
        # Top game đánh giá cao nhất của user
        top_rated_games = sorted(history_items, key=lambda x: x["user_rating"], reverse=True)[:5]
        
        return {
            "status": "success",
            "user_id": user_id,
            "total_interactions": total_reviews,
            "average_rating": round(avg_rating, 2),
            "rating_distribution": rating_dist,
            "favorite_categories": dict(sorted_categories[:5]),
            "top_categories": top_categories,
            "gamer_persona": persona,
            "favorite_games": top_rated_games,
        }

analytics_tool = AnalyticsTool(df_interactions, cb_model)
print("[*] AnalyticsTool initialized successfully!")

---

## 3. Xây dựng Agent Core & Prompt Orchestrator (Intent Routing & ReAct Loop)

AI Agent đóng vai trò là **Trợ lý Chuyên gia Gaming AI**, sở hữu khả năng:
1. **Phân tích câu hỏi tự nhiên** của người dùng để nhận diện Intent:
   - `INTENT_ANALYZE`: Yêu cầu phân tích lịch sử/hồ sơ (`AnalyticsTool`).
   - `INTENT_RECOMMEND`: Yêu cầu gợi ý game (`RecommendTool`).
   - `INTENT_EXPLAIN`: Yêu cầu giải thích tại sao gợi ý (`ExplainTool`).
   - `INTENT_CHAT`: Chào hỏi, trò chuyện kiến thức game.
2. **Thực thi Tool** và tổng hợp câu trả lời theo phong cách chuyên gia: thân thiện, sinh động, chuẩn xác và thuyết phục.

In [ ]:
class GameAgentRunner:
    """
    AI Agent Điều Phối Trung Tâm (ReAct Orchestration & Tool Routing).
    """

    SYSTEM_PROMPT = """Bạn là AI Gaming Assistant - Trợ lý Chuyên gia Trí tuệ Nhân tạo về Trò chơi Điện tử.
Nhiệm vụ của bạn là tư vấn, phân tích hồ sơ game thủ, gợi ý những tựa game đỉnh cao và giải thích lý do đề xuất một cách minh bạch, hấp dẫn.
Luôn giữ văn phong chuyên nghiệp, nhiệt huyết, am hiểu sâu sắc về thế giới game."""

    def __init__(
        self,
        recommend_tool: RecommendTool,
        explain_tool: ExplainTool,
        analytics_tool: AnalyticsTool,
    ):
        self.recommend_tool = recommend_tool
        self.explain_tool = explain_tool
        self.analytics_tool = analytics_tool
        self.history: List[Dict[str, str]] = []

    def route_intent(self, message: str, user_id: Optional[str] = None) -> Dict[str, Any]:
        """
        Xác định ý định của người dùng bằng Pattern & Entity Matching.
        """
        msg_lower = message.lower()
        
        # 1. Ý định phân tích hồ sơ
        if any(kw in msg_lower for kw in ["phân tích", "hồ sơ", "gu game", "thống kê", "lịch sử của tôi", "persona", "thói quen"]):
            return {"intent": "ANALYZE", "user_id": user_id}
            
        # 2. Ý định giải thích
        elif any(kw in msg_lower for kw in ["tại sao", "vì sao", "giải thích", "lý do", "có gì hay", "đánh giá thế nào"]):
            clean_query = re.sub(r"(tại sao|vì sao|giải thích|lý do|gợi ý|cho tôi|được|lại|tựa|game|này|nhỉ|hả)", "", msg_lower).strip(" ?:,'\"")
            return {"intent": "EXPLAIN", "query": clean_query, "user_id": user_id}
            
        # 3. Ý định gợi ý game
        elif any(kw in msg_lower for kw in ["gợi ý", "đề xuất", "tìm", "recommend", "chơi gì", "tư vấn", "game nào", "thích"]):
            k_match = re.search(r"(\d+)\s*(game|tựa game|trò chơi)", msg_lower)
            top_k = int(k_match.group(1)) if k_match else 5
            
            clean_kw = re.sub(r"(gợi ý|đề xuất|tìm|kiếm|cho tôi|\d+\s*game|tựa game|hay nhất|phù hợp|nhé|nào|với tôi|chơi gì)", "", msg_lower).strip(" ?:,'\"")
            return {"intent": "RECOMMEND", "query": clean_kw if len(clean_kw) > 2 else None, "top_k": top_k, "user_id": user_id}
            
        # 4. Trò chuyện thông thường
        else:
            return {"intent": "CHAT", "message": message}

    def handle_message(self, message: str, user_id: Optional[str] = None) -> str:
        """
        Tiếp nhận thông điệp, điều phối Tool và sinh câu trả lời hoàn chỉnh.
        """
        route = self.route_intent(message, user_id=user_id)
        intent = route["intent"]
        
        # --- XỬ LÝ INTENT: ANALYZE ---
        if intent == "ANALYZE":
            if not user_id:
                return "🤖 **AI Gaming Assistant**: Bạn vui lòng cung cấp mã người dùng (`user_id`) để tôi có thể truy xuất và phân tích hồ sơ chơi game của bạn nhé!"
            
            res = self.analytics_tool.run(user_id)
            if res["status"] == "not_found":
                return f"🤖 **AI Gaming Assistant**: {res['message']}"
                
            top_cats_str = ", ".join([f"**{k}** ({v} lượt)" for k, v in list(res["favorite_categories"].items())[:3]])
            fav_games_str = "\n".join([f"  - 🎮 **{g['title']}** (Đánh giá: ⭐ {g['user_rating']}/5.0 | Thể loại: *{g['category']}*)" for g in res["favorite_games"][:3]])
            
            reply = (
                f"🎮 **HỒ SƠ PHÂN TÍCH GAME THỦ (Gamer Profile Analysis)**\n\n"
                f"- 👤 **Mã Game thủ:** `{user_id}`\n"
                f"- 🏆 **Phong cách Gamer Persona:** 🎯 **{res['gamer_persona']}**\n"
                f"- 📊 **Tổng số game đã trải nghiệm:** **{res['total_interactions']}** game\n"
                f"- ⭐ **Điểm đánh giá trung bình:** **{res['average_rating']} / 5.0**\n"
                f"- 🕹️ **Thể loại đam mê nhất:** {top_cats_str}\n\n"
                f"🔥 **Top tựa game bạn yêu thích nhất trong lịch sử:**\n{fav_games_str}\n\n"
                f"💡 *Nhận định AI:* Bạn là một game thủ có gu thẩm mỹ rõ rệt và sự gắn bó cao với các tựa game chất lượng đỉnh cao. "
                f"Tôi có thể gợi ý ngay những game mới phù hợp chuẩn xác với phong cách này của bạn!"
            )
            return reply

        # --- XỬ LÝ INTENT: RECOMMEND ---
        elif intent == "RECOMMEND":
            query = route.get("query")
            top_k = route.get("top_k", 5)
            
            res = self.recommend_tool.run(user_id=user_id, query=query, top_k=top_k, diversity_weight=0.3)
            recs = res.get("recommendations", [])
            
            if not recs:
                return "🤖 **AI Gaming Assistant**: Rất tiếc, tôi chưa tìm thấy tựa game nào hoàn toàn khớp với yêu cầu. Bạn hãy thử mô tả cụ thể hơn nhé!"
                
            items_str_list = []
            for idx, item in enumerate(recs, 1):
                title = item.get("title", "Unknown")
                cat = item.get("category", "Video Games")
                rating = item.get("avg_rating", 4.5)
                score = item.get("hybrid_score", 0.0)
                exp = item.get("explanation", {})
                quote = exp.get("social_proof_quote", "")
                reasons = exp.get("reasons", [])
                reason_str = f" ({reasons[0]})" if reasons else ""
                
                block = (
                    f"**{idx}. 🎮 {title}**\n"
                    f"   - 🏷️ Thể loại: *{cat}* | ⭐ Đánh giá: **{rating:.1f}/5.0** | 🎯 Hybrid Match: **{score:.1%}**{reason_str}\n"
                    f"   - 💬 *Đánh giá từ cộng đồng:* \"{quote[:130]}...\""
                )
                items_str_list.append(block)
                
            mode_desc = f"cá nhân hóa cho tài khoản `{user_id}`" if user_id else f"dựa theo tìm kiếm: *'{query}'*"
            reply = (
                f"🚀 **DANH SÁCH GAME ĐƯỢC AI ĐỀ XUẤT ({mode_desc})**\n\n" +
                "\n\n".join(items_str_list) +
                f"\n\n✨ *Mẹo:* Bạn có thể hỏi: *'Tại sao lại gợi ý tựa game số 1 cho tôi?'* để nghe phân tích sâu hơn!"
            )
            return reply

        # --- XỬ LÝ INTENT: EXPLAIN ---
        elif intent == "EXPLAIN":
            query_game = route.get("query", "")
            res = self.explain_tool.run(item_asin_or_title=query_game, user_id=user_id)
            
            if res["status"] == "error":
                return f"🤖 **AI Gaming Assistant**: {res['message']}"
                
            reasons_bullets = "\n".join([f"  - 🔹 {r}" for r in res["key_reasons"]])
            anchor_info = f"\n  - 🔗 **Tựa game mỏ neo tương đồng nhất:** *{res['anchor_game']}* (Độ tương đồng: **{res['anchor_similarity_pct']}%**)" if res.get("anchor_game") else ""
            
            reply = (
                f"🔍 **GIẢI THÍCH CHI TIẾT ĐỀ XUẤT GAME: {res['title']}**\n\n"
                f"- 🏷️ **Thể loại:** *{res['category']}*\n"
                f"- ⭐ **Đánh giá cộng đồng:** **{res['average_rating']} / 5.0**\n"
                f"- 🧠 **Các căn cứ logic đề xuất:**\n{reasons_bullets}{anchor_info}\n\n"
                f"💬 **Trích dẫn nhận xét thực tế từ người chơi:**\n"
                f"> \"{res['social_proof_quote']}\"\n\n"
                f"🏆 **Kết luận của AI:** Đây là tựa game vô cùng sáng giá và có độ khớp cao với sở thích của bạn!"
            )
            return reply

        # --- INTENT: CHAT ---
        else:
            return (
                f"🤖 **AI Gaming Assistant**: Xin chào! Tôi là Trợ lý AI Chuyên gia Gợi ý & Phân tích Game.\n\n"
                "Tôi có thể giúp bạn:\n"
                "1. 📊 **Phân tích hồ sơ game thủ & gu chơi game của bạn** (Ví dụ: *'Hãy phân tích lịch sử của tôi'*)\n"
                "2. 🎯 **Gợi ý game cá nhân hóa hoặc theo sở thích** (Ví dụ: *'Gợi ý cho tôi 5 game nhập vai đỉnh nhất'*)\n"
                "3. 💡 **Giải thích tường tận lý do đề xuất một tựa game** (Ví dụ: *'Vì sao lại đề xuất game này?'*)\n\n"
                "Hãy cho tôi biết bạn đang muốn khám phá điều gì nhé!"
            )

agent_runner = GameAgentRunner(recommend_tool, explain_tool, analytics_tool)
print("[*] GameAgentRunner initialized successfully!")

---

## 4. Thử Nghiệm Kịch Bản Thực Tế (Live Simulation & Validation)

### 🧪 Kịch bản 1: Phân tích Hồ sơ & Gamer Persona của User thực tế

In [ ]:
# Lấy một User tích cực từ tập interactions
sample_user_id = df_interactions.group_by("user_id").len().sort("len", descending=True)[0, "user_id"]
print(f"Selected sample user: {sample_user_id}")

query_1 = "Phân tích gu chơi game và lịch sử đánh giá của tôi"
response_1 = agent_runner.handle_message(query_1, user_id=sample_user_id)
print(response_1)

### 🧪 Kịch bản 2: Gợi ý Cá nhân hóa & Đa dạng hóa (Personalized Recommendation)

In [ ]:
query_2 = "Hãy gợi ý cho tôi 3 tựa game đỉnh cao phù hợp nhất với tôi nhé!"
response_2 = agent_runner.handle_message(query_2, user_id=sample_user_id)
print(response_2)

### 🧪 Kịch bản 3: Gợi ý cho Cold-Start User theo Thể loại / Mô tả Ngôn ngữ Tự nhiên

In [ ]:
query_3 = "Tìm cho tôi 3 game Mario phiêu lưu hay nhất"
response_3 = agent_runner.handle_message(query_3, user_id=None) # Cold-Start User
print(response_3)

### 🧪 Kịch bản 4: Giải thích Chi Tiết Một Đề Xuất Cụ Thể (Deep-dive Explanation)

In [ ]:
# Yêu cầu giải thích tựa game Mario
query_4 = "Tại sao lại giải thích và gợi ý Mario cho tôi?"
response_4 = agent_runner.handle_message(query_4, user_id=sample_user_id)
print(response_4)

---

## 5. Đánh Giá & Tổng Kết Bước Thử Nghiệm (Task 7.1 Summary)

| Tiêu chí | Kết quả thử nghiệm trên Notebook | Trạng thái |
| :--- | :--- | :---: |
| **RecommendTool** | Tích hợp hoàn hảo Hybrid Recommender + MMR Re-ranking, hỗ trợ cả User ID & Cold-Start query | ✅ Hoạt động chuẩn xác |
| **ExplainTool** | Trích xuất tín hiệu đa chiều (Anchor similarity, Genre overlap, Community consensus, Real quote) | ✅ Minh bạch, thuyết phục |
| **AnalyticsTool** | Tính toán phân bố rating, top thể loại và xác định Gamer Persona tự động | ✅ Đầy đủ, sâu sắc |
| **Intent Router & Orchestrator** | Phân loại câu hỏi tự nhiên chính xác, điều phối tool mượt mà, phản hồi giàu thông tin | ✅ Sẵn sàng đóng gói `src/agent/` |